In [1]:
import glob
import time
import cv2
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)
print(torch.__version__)

import json
from pathlib import Path
from enum import Enum
from typing import List
from ultralytics import YOLO

cuda
2.1.2+cu118


In [2]:
KEYPOINT_IDS = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

class Keypoint:
    def __init__(self, id, x, y):
        self.id = id
        self.pixelPosition = (x, y)

class PoseResult:
    def __init__(self, file, keypoints):
        self.file = file
        self.keypoints = keypoints

In [3]:
import numpy as np
from typing import List

def processKeypoints(pose) -> List[Keypoint]:
    """Convert Ultralytics pose result -> List[Keypoint] with normalized coords (0..1)."""
    kpts = pose.keypoints
    if kpts is None or len(kpts) == 0:
        return []

    # choose best detected person if multiple
    best_i = 0
    if pose.boxes is not None and pose.boxes.conf is not None and len(pose.boxes.conf) > 0:
        best_i = int(np.argmax(pose.boxes.conf.cpu().numpy()))

    # normalized coords [num_people, 17, 2] -> take best -> [17,2]
    kpt_xyn = kpts.xyn[best_i].cpu().numpy()

    final_keypoints: List[Keypoint] = []
    for i, (x, y) in enumerate(kpt_xyn):
        final_keypoints.append(Keypoint(KEYPOINT_IDS[i], float(x), float(y)))

    return final_keypoints



def savePoseResultsToJson(results: List[PoseResult], output_path: str, model: str):
    keypoints_dict = {
        "model": model,
        "keypointsOnImages": [
            {
                "file": result.file,
                "keypoints": [
                    {
                        "id": keypoint.id,
                        "pixelPosition": {
                            "x": keypoint.pixelPosition[0],
                            "y": keypoint.pixelPosition[1],
                        },
                    }
                    for keypoint in result.keypoints
                ],
            }
            for result in results
        ],
    }

    with open(output_path, "w") as f:
        json.dump(keypoints_dict, f, indent=4)

In [4]:
TITLE='test-blindoff-dataset'
INPUT_DIR = f"D:\\Magistrska\\blindoff-magistrska\\fitcode-frontend-next\\public\\exercise-cut-videos-to-images\\{TITLE}\\images"
OUTPUT_DIR = f"D:\\Magistrska\\blindoff-magistrska\\fitcode-frontend-next\\public\\exercise-cut-videos-to-images\\{TITLE}\\results"
SAVE_VIS = False # True to save visualizations (with keypoint drawings)
SHOW_VIS = False # True to show visualizations
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.tif", "*.tiff")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

yolo_path = r"D:\Magistrska\blindoff-magistrska\notebooks\runs\pose\train\weights\best.pt"
pose_model = YOLO(yolo_path)


# ----------------- Collect images -----------------
input_dir = Path(INPUT_DIR)
assert input_dir.exists(), f"INPUT_DIR does not exist: {input_dir}"

image_paths = []
for ext in IMG_EXTS:
    image_paths.extend(glob.glob(str(input_dir / ext)))
image_paths = sorted(image_paths)

if not image_paths:
    raise RuntimeError(f"No images found in {input_dir} with extensions: {IMG_EXTS}")

# ----------------- Output dir -----------------
out_dir = Path(OUTPUT_DIR)
if SAVE_VIS:
    out_dir.mkdir(parents=True, exist_ok=True)

# ----------------- Process loop -----------------
total = len(image_paths)
start_time = time.time()
max_fps = 0.0

pose_results = []

for idx, img_path in enumerate(image_paths, start=1):
    print(f"Processing image ({idx}/{total})")
    frame = cv2.imread(img_path)
    if frame is None:
        print(f"[WARN] Could not read image: {img_path}")
        continue

    # --------- Pose inference ---------
    model_pose_results = pose_model(frame)

    # --------- Visualization ---------
    vis_frame = frame.copy()
    pose = model_pose_results[0]

    w, h = frame.shape[1], frame.shape[0]
    keypoints = processKeypoints(pose)
    pose_results.append(PoseResult(file=Path(img_path).name, keypoints=keypoints))


# ----------------- Done -----------------
print(f"Processed: {idx}/{total} images")

savePoseResultsToJson(
    pose_results,
    output_path=str(out_dir / "YOLO256_trained_on_blindoff_dataset.json"),
    model="YOLO256_trained_on_blindoff_dataset",
)

if SHOW_VIS:
    cv2.destroyAllWindows()

cuda
Processing image (1/944)

0: 256x192 1 person, 145.1ms
Speed: 58.3ms preprocess, 145.1ms inference, 5.2ms postprocess per image at shape (1, 3, 256, 192)
Processing image (2/944)

0: 256x192 1 person, 30.6ms
Speed: 1.3ms preprocess, 30.6ms inference, 4.0ms postprocess per image at shape (1, 3, 256, 192)
Processing image (3/944)

0: 256x192 1 person, 24.9ms
Speed: 4.8ms preprocess, 24.9ms inference, 6.6ms postprocess per image at shape (1, 3, 256, 192)
Processing image (4/944)

0: 256x192 1 person, 47.2ms
Speed: 1.4ms preprocess, 47.2ms inference, 7.2ms postprocess per image at shape (1, 3, 256, 192)
Processing image (5/944)

0: 256x192 1 person, 25.5ms
Speed: 2.2ms preprocess, 25.5ms inference, 3.6ms postprocess per image at shape (1, 3, 256, 192)
Processing image (6/944)

0: 256x192 1 person, 35.4ms
Speed: 1.1ms preprocess, 35.4ms inference, 3.4ms postprocess per image at shape (1, 3, 256, 192)
Processing image (7/944)

0: 256x192 1 person, 28.7ms
Speed: 1.9ms preprocess, 28.7ms 